# Coronary Segmentation — Kaggle BUILD notebook

nnU-Net **teacher** → distill **TinyU-Net student** → export ONNX + state_dict, on a Kaggle GPU. Same pipeline as the Colab version, Kaggle-native paths.

### Kaggle setup
1. **Settings → Accelerator → GPU**; **Internet → ON**.
2. **Add Input → Datasets**: attach **ARCADE** (contains `syntax/`) and **DCA1** (the .pgm files). They mount under `/kaggle/input/`.
3. Edit `ARCADE_INPUT` / `DCA1_INPUT` to the real paths, keep `DRY_RUN = True`, **Run All**.

Note: nnU-Net state lives under `/kaggle/working` (persists within a session; **Save Version** to keep it). Kaggle sessions are time-boxed — keep `DRY_RUN` on first.

## 1 · GPU + code + install

In [ ]:
import os, sys, torch, glob
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
TORCH = torch.__version__.split('+')[0]     # pin Kaggle's torch so pip can't swap/break it
                                            # (prevents 'duplicate registrations for aten.linspace')
GITHUB_TOKEN = ''                                             # '' if PUBLIC else a PAT
REPO_SLUG = 'jugalmodi0111/interventional-imaging-pipeline'
REPO = '/kaggle/working/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install nnunetv2 monai SimpleITK onnx onnxscript onnxruntime \
    scikit-image scikit-learn pycocotools pyyaml tqdm "torch=={TORCH}" "numpy<2.1"
from src import env
E = env.setup()                                              # sets nnU-Net env vars under /kaggle/working
print('build env ready | torch', torch.__version__)

## 2 · Attach data → symlink into data/raw + DRY_RUN

In [ ]:
import glob, os
# auto-discover ARCADE root (contains syntax/) and the DCA1 .pgm folder, wherever Kaggle nested them
sj = glob.glob('/kaggle/input/**/syntax/*/annotations/*.json', recursive=True)
pg = glob.glob('/kaggle/input/**/*.pgm', recursive=True)
assert sj, 'No ARCADE syntax json under /kaggle/input — attach the ARCADE dataset.'
assert pg, 'No DCA1 .pgm under /kaggle/input — attach the DCA1 dataset.'
ARCADE_INPUT = sj[0].split('/syntax/')[0]          # parent of syntax/
DCA1_INPUT   = os.path.dirname(pg[0])              # folder holding the .pgm pairs
DRY_RUN = True
print('ARCADE_INPUT =', ARCADE_INPUT); print('DCA1_INPUT   =', DCA1_INPUT)
os.makedirs('data/raw', exist_ok=True)
!ln -sfn {ARCADE_INPUT} data/raw/arcade
!ln -sfn {DCA1_INPUT}   data/raw/dca1
assert glob.glob('data/raw/arcade/syntax/**/*.json', recursive=True)
assert glob.glob('data/raw/dca1/**/*.pgm', recursive=True)
print('OK: ARCADE syntax json + DCA1 pgm present')

## 3 · Standardize → binary vessel pairs + nnU-Net raw

In [ ]:
!python -m src.data_prep.arcade_to_coco --config configs/coronary_seg.yaml
!python -m src.data_prep.dca1_to_nnunet --config configs/coronary_seg.yaml
print('processed pairs:', len(glob.glob('data/processed/coronary/img/*.png')))

## 4 · nnU-Net teacher (2D) + cache logits

In [ ]:
DATASET_ID = '001'
# Kaggle GPU sessions cap at ~12h. 250 epochs won't fit -> use 100 (~4-5h on P100/T4). --c resumes if it drops.
TRAINER = 'nnUNetTrainer_5epochs' if DRY_RUN else 'nnUNetTrainer_100epochs'
CACHE = f"{E['root']}/teacher_cache/coronary"; os.makedirs(CACHE, exist_ok=True)
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity
!nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER} --c || nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER}
# build predict as one f-string line — IPython {VAR} substitution breaks across '\' continuations
pred = (f"nnUNetv2_predict -i {os.environ['nnUNet_raw']}/Dataset{DATASET_ID}_Coronary/imagesTr "
        f"-o {CACHE} -d {DATASET_ID} -c 2d -f 0 -tr {TRAINER} --save_probabilities")
!{pred}
print('teacher npz:', len(glob.glob(f'{CACHE}/*.npz')))

## 5 · Distill TinyU-Net student

In [ ]:
import yaml
from torch.utils.data import DataLoader
from src.models.seg_student import TinyUNet
from src.models.distill import distill, TeacherCacheDataset
from src.eval.metrics import dice, cldice
cfg = yaml.safe_load(open('configs/coronary_seg.yaml'))
ds = TeacherCacheDataset('data/processed/coronary', CACHE, size=cfg['preprocess']['size'])
loader = DataLoader(ds, batch_size=cfg['train']['batch'], shuffle=True, num_workers=2)
student = TinyUNet(in_ch=1, n_classes=1, base=cfg['student']['base_ch'], depth=cfg['student'].get('depth',4))
def quick_eval(m):
    m.eval()
    with torch.no_grad():
        x,y,_ = ds[0]; p = torch.sigmoid(m(x[None].cuda())).cpu().numpy().squeeze() >= 0.5
    m.train(); g = y.numpy().squeeze() > 0.5
    return f'Dice {dice(p,g):.3f} clDice {cldice(p,g):.3f}'
RUNS = f"{E['runs']}/coronary"; os.makedirs(RUNS, exist_ok=True)
distill(student, loader, epochs=(3 if DRY_RUN else cfg['train']['epochs']), lr=cfg['train']['lr'],
        alpha=cfg['distill']['alpha'], T=cfg['distill']['temperature'],
        eval_fn=quick_eval, ckpt=f'{RUNS}/student.pt')

## 6 · Export ONNX + state_dict (handoff to Mac CoreML)

In [ ]:
!python -m src.export.to_onnx --weights {RUNS}/student.pt --out {RUNS}/student.onnx
!python -m src.export.quantize_int8 --onnx {RUNS}/student.onnx
!python -m src.eval.edge_benchmark --model {RUNS}/student.onnx
print('\nDownload from output:', f'{RUNS}/student.pt', '(-> Mac: make export-coreml)')